In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from CADETProcess.processModel import (
    ComponentSystem, FlowSheet, Inlet, Langmuir, Outlet, Process, TubularReactor,
)
from CADETProcess.simulator import Cadet

simulator = Cadet()


def tubing_pulse(ncol=50):
    """2 mL tubing at 6 mL/min with a 5 s tracer pulse, discretized with ncol axial cells."""
    component_system = ComponentSystem(["Tracer"])

    inlet = Inlet(component_system, name="inlet")
    inlet.flow_rate = 6e-6 / 60  # 6 mL/min in m³/s

    tubing = TubularReactor(component_system, name="tubing")
    tubing.length = 1.0
    tubing.diameter = np.sqrt(4 * 2e-6 / (np.pi * tubing.length))  # 2 mL volume
    tubing.axial_dispersion = 1e-8
    tubing.discretization.ncol = ncol

    outlet = Outlet(component_system, name="outlet")

    flow_sheet = FlowSheet(component_system)
    flow_sheet.add_unit(inlet)
    flow_sheet.add_unit(tubing)
    flow_sheet.add_unit(outlet)
    flow_sheet.add_connection(inlet, tubing)
    flow_sheet.add_connection(tubing, outlet)

    process = Process(flow_sheet, "tubing_pulse")
    process.cycle_time = 60
    process.add_event("pulse_on", "flow_sheet.inlet.c", [1], 0)
    process.add_event("pulse_off", "flow_sheet.inlet.c", [0], 5)
    return process

# 99.6 Common Problems

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Attribute changes, simulation calls, `try`/`except` and plotting.

**Introduces:** numerical dispersion and grid checks, output time resolution, units, per-component values, event times, kinetic binding, porosity, notebook state, and pitfalls in fitting and optimization.

1. Simulation results
2. Model setup
3. Notebook issues
4. Fitting and optimization

Each problem is described by its **symptom**, its **cause**, how to **check** it and how to **fix** it. The examples use a 2 mL tubing with a 5 s tracer pulse at 6 mL/min. This page is a reference: look up a problem when it occurs, or browse it after the main chapters.

## Simulated peaks are too wide: numerical dispersion

| | |
|---|---|
| **Symptom** | Peaks or fronts are wider and lower than expected, and change when the grid changes. |
| **Cause** | The model is solved on a finite number of axial cells (`discretization.ncol`). A coarse grid spreads the signal on its own: **numerical dispersion**. |
| **Check** | Simulate again with twice the number of cells. If the curve changes visibly, the grid is too coarse. |
| **Fix** | Increase `ncol` until the result no longer changes (**grid convergence**). |

The tubing has very little physical dispersion (1e-8 m²/s), so with a fine grid the 5 s pulse should leave almost unchanged. The same model is simulated with 10, 50 and 200 axial cells:

In [ ]:
# Simulate the same tubing with three grid resolutions
coarse_results = simulator.simulate(tubing_pulse(ncol=10))
medium_results = simulator.simulate(tubing_pulse(ncol=50))
fine_results = simulator.simulate(tubing_pulse(ncol=200))

In [ ]:
# Compare the outlet signals and their peak heights
fig, ax = plt.subplots(figsize=(8, 3))
for label, results, style in [
    ("10 cells", coarse_results, ":"),
    ("50 cells", medium_results, "--"),
    ("200 cells", fine_results, "-"),
]:
    signal = results.solution.outlet.outlet
    concentration = signal.solution[:, 0]  # column 0: the only component, Tracer
    ax.plot(signal.time, concentration, linestyle=style, linewidth=2, label=label)
    print(f"{label}: peak {concentration.max():.2f} mol/m³")
ax.set(xlim=(10, 40), xlabel="Time / s", ylabel="Tracer / mol/m³")
_ = ax.legend()
plt.show()

With 10 cells, the peak reaches only about half the feed concentration; from 50 cells on it stays close to the feed concentration of 1 mol/m³. The **mean** arrival time is about 22.5 s for all three grids (20 s residence time plus half of the 5 s pulse): the grid changes the shape, not the delay. Fits of dispersion are therefore sensitive to the grid; fits of porosity from the mean delay are not.

## Peaks look stepped or too early: output time resolution

| | |
|---|---|
| **Symptom** | Curves look angular; peak times come out as whole seconds. |
| **Cause** | The simulator stores the solution every `time_resolution` seconds, **1 s** by default. The calculation itself is accurate; only the stored points are coarse. |
| **Check** | `simulator.time_resolution` |
| **Fix** | Use a smaller value for short pulses or sharp peaks, e.g. `simulator.time_resolution = 0.1`. More points make results larger and slower to handle. |

In [ ]:
# Peak time with the default and a finer output resolution
print("Default resolution:", simulator.time_resolution, "s")
signal = fine_results.solution.outlet.outlet
print(f"  peak at {signal.time[signal.solution[:, 0].argmax()]:.1f} s")

fine_output = Cadet()
fine_output.time_resolution = 0.1
signal = fine_output.simulate(tubing_pulse(ncol=200)).solution.outlet.outlet
print(f"Resolution 0.1 s: peak at {signal.time[signal.solution[:, 0].argmax()]:.1f} s")

## Nothing seems to happen, or everything is too fast: units

| | |
|---|---|
| **Symptom** | Breakthrough after a few seconds, or never; concentrations far off. |
| **Cause** | CADET-Process uses **SI units**: m, m², m³/s, s, mol/m³ (= mM). Laboratory values come in mL/min, cm, min or g/L. |
| **Check** | Compute one familiar number by hand, e.g. residence time = volume / flow rate. |
| **Fix** | Convert once, in a clearly commented line. |

| Laboratory value | SI value in CADET-Process |
|---|---|
| 6 mL/min | `6e-6 / 60` m³/s = 1e-7 m³/s |
| 10 cm column length | `0.1` m |
| 2 cm column diameter | `0.02` m |
| 30 min gradient | `30 * 60` s |
| 1 mM | `1` mol/m³ |
| slope of 500 mM in 30 min | `500 / (30 * 60)` mol/m³ per s |

In [ ]:
# Check: 2 mL of tubing at 6 mL/min should take 20 s
process = tubing_pulse()
volume = process.flow_sheet.tubing.volume  # m³
flow_rate = process.flow_sheet.inlet.flow_rate[0]  # m³/s, first polynomial coefficient
print(f"Residence time: {volume / flow_rate:.1f} s")

## `ValueError: Expected size 3`: one value per component

| | |
|---|---|
| **Symptom** | Setting a parameter raises `ValueError: Expected size …`. |
| **Cause** | Many parameters need **one value per component** (or per bound state), in the order of the `ComponentSystem`. In the SMA model, salt is the first component. |
| **Check** | `print(binding_model.parameters)` and `component_system.names` |
| **Fix** | Give exactly one value per component, also for components that do not bind (e.g. `0`). |

In [ ]:
# A list that is too short, for three components
component_system = ComponentSystem(["A", "B", "C"])
binding_model = Langmuir(component_system, name="langmuir")
try:
    binding_model.capacity = [10, 10]
except ValueError as error:
    print("ValueError:", error)
binding_model.capacity = [10, 10, 10]
print("capacity:", binding_model.capacity)

The message does not name the parameter: the traceback line in your own code does. [01.4 Contributing](../../01%20Introduction/01.4%20Contributing/) uses this example to write an Issue.

## Events at the wrong time: event times and `cycle_time`

| | |
|---|---|
| **Symptom** | A step or gradient starts at an unexpected time, or a rerun cell raises `CADETProcessError: Event already exists`. |
| **Cause** | Event times are taken **modulo the cycle time**: an event at 90 s in a 60 s cycle takes place at 30 s, without an error. Adding an event with an existing name to the same process fails. |
| **Check** | Print the events with their times; compare with `process.cycle_time`. |
| **Fix** | Set `cycle_time` first and keep all event times below it. Create the process and its events in one cell, so rerunning the cell starts from a fresh process. |

In [ ]:
# An event after the end of the cycle is moved into the cycle
process = tubing_pulse()
process.add_event("late", "flow_sheet.inlet.c", [0], 90)
print("cycle_time:", process.cycle_time, "s")
for event in process.events:
    print(f"{event.name}: {event.time} s")

# Adding an event with an existing name again
try:
    process.add_event("pulse_on", "flow_sheet.inlet.c", [1], 0)
except Exception as error:
    print(type(error).__name__ + ":", error)

## Binding looks wrong: kinetic or equilibrium binding

| | |
|---|---|
| **Symptom** | Breakthrough comes earlier or more gradually than expected from the isotherm; simulations with large rates are slow. |
| **Cause** | Binding models are **kinetic by default** (`is_kinetic = True`): `adsorption_rate` and `desorption_rate` then set the speed of binding, not only its equilibrium. Small rates delay binding; very large rates make the equations stiff. |
| **Check** | `binding_model.is_kinetic`; compare the rates with the residence time. |
| **Fix** | For rapid equilibrium, set `is_kinetic = False` instead of using very large rates. Only the ratio of the rates then matters. See [02.5 Binding](../../02%20CADET%20Fundamentals/02.5%20Binding/). |

In [ ]:
print("Default is_kinetic:", binding_model.is_kinetic)

## Porosity does not match: which porosity?

| | |
|---|---|
| **Symptom** | A fitted porosity is implausibly high, or two tracers give different porosities. |
| **Cause** | Mixer and tubing delay the signal too; assigning that **system delay** to the column inflates its porosity. Tracers also differ: Acetone enters the particle pores (total porosity), Dextran does not (bed porosity). |
| **Check** | Measure a column-bypass pulse; note which tracer was used. |
| **Fix** | Subtract the system delay or model mixer and tubing ([02.4 Residence Time Distributions](../../02%20CADET%20Fundamentals/02.4%20Residence%20Time%20Distributions/)); use the porosity that matches the tracer and column model ([01.5 Glossary](../../01%20Introduction/01.5%20Glossary/)). |

## Notebook issues

| Symptom | Cause | Fix |
|---|---|---|
| A cell uses old values, or a changed parameter has no effect | Cells were run out of order; a name points to the object assigned last | *Kernel → Restart Kernel and Run All Cells* |
| A widget shows an old process | The process was changed in code after the widget was drawn | Run the widget cell again |
| A widget shows only "Loading widget…" | The browser has a cached, outdated version of the widget code | Reload the page without cache (**Ctrl+Shift+R**), then restart the kernel |
| No figures appear after the first plot | The inline plotting backend was first activated inside a CADET-Process plot | Run `%matplotlib inline` at the start of the notebook |

## Fitting and optimization

| Symptom | Cause | Fix |
|---|---|---|
| A fitted parameter sits at its bound | The bound is too narrow, or the model is missing an effect | Widen the bound once; if it stays there, revisit the model |
| Different start values give different results | Local optimizer and several minima | Try several starts, or a population-based optimizer ([99.4 Optimization Basics](../99.4%20Optimization%20Basics/)) |
| An uncalibrated signal fits badly in amplitude | The detector signal is not a concentration | Compare the curve shape with `Shape` instead of `SSE`/`RMSE` ([04.2 Parameter Fitting](../../04%20Characterization%20and%20Optimization/04.2%20Parameter%20Fitting/)) |
| Dispersion cannot be fitted from a delay | The mean delay only depends on volumes and flow | Fit the full curve ([04.2 Parameter Fitting](../../04%20Characterization%20and%20Optimization/04.2%20Parameter%20Fitting/)) |
| Every candidate violates the purity constraint slightly | Fractionation meets its requirement only within a tolerance | Require slightly more in the fractionation (0.951) than in the constraint (0.95) ([04.4 Process Optimization](../../04%20Characterization%20and%20Optimization/04.4%20Process%20Optimization/)) |
| The search stops at its iteration limit | The budget ran out | An iteration limit is not convergence: check the convergence plot and the selected curve or chromatogram |
| Performance of a cyclic process changes from cycle to cycle | The process has not reached cyclic stationarity | Simulate until stationarity before evaluating purity, recovery and productivity ([04.4 Process Optimization](../../04%20Characterization%20and%20Optimization/04.4%20Process%20Optimization/)) |

## Summary

- Check the grid by refining it, and the output by refining `time_resolution`.
- Use SI units, one value per component, and event times below `cycle_time`.
- Restart the kernel and run all cells when results do not match the code.
- In fitting and optimization, bounds, start values, tolerances and stationarity decide whether a result can be trusted.

**Further reading:**
- von Lieres, E.; Andersson, J. (2010): A fast and accurate solver for the general rate model of column liquid chromatography. *Computers & Chemical Engineering* 34 (8), 1180–1191. [doi:10.1016/j.compchemeng.2010.03.008](https://doi.org/10.1016/j.compchemeng.2010.03.008)
- Breuer, J. M.; Leweke, S.; Schmölder, J.; Gassner, G.; von Lieres, E. (2023): Spatial discontinuous Galerkin spectral element method for a family of chromatography models in CADET. *Computers & Chemical Engineering* 177, 108340. [doi:10.1016/j.compchemeng.2023.108340](https://doi.org/10.1016/j.compchemeng.2023.108340)